## Install requirements

- tested on python 3.12.8

### basic setup

In [ ]:
%pip install --upgrade pip setuptools wheel

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

parent_dir = os.path.dirname(os.path.abspath('./'))
os.chdir(parent_dir)
print(parent_dir)

In [ ]:
%pip install -r "requirements (no version).txt" --force-reinstall  # if you want to install the latest versions
# %pip install -r "requirements (now version).txt" --force-reinstall  # if you want to install the exact versions that we used for our experiments
%pip cache purge

!conda update --all --yes
!conda clean --all --yes

  Using cached numpy-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached scikit_learn-1.6.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached scipy-1.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached torch-2.6.0-cp312-cp312-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached pillow-11.1.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cach

### test torch

In [1]:
%pip show torch

Name: torch
Version: 2.7.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /home/yoom618/.conda/envs/py312/lib/python3.12/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, causal_conv1d, lightning, mamba_ssm, pytorch-lightning, torchaudio, torchmetrics, torchvision
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

torch.cuda.is_available()

True

### install mamba-ssm and causal-conv1d
- install causal-conv1d and mamba-ssm manually since they take a long time to install

- You may have trouble while installing those libraries due to various reasons(e.g. GPU compatibility, mismatched versions, ...). In that case, we recommend to find the error message on the [mamba's GitHub issues](https://github.com/state-spaces/mamba/issues).
    - For instance, you may have to add `--no-cache-dir` or `--no-build-isolation` flags to the pip install command. [[ref1](https://github.com/state-spaces/mamba?tab=readme-ov-file#installation), [ref2](https://github.com/state-spaces/mamba/issues/794)]

    - Or you may have to install via git clone instead of pip install. [[ref3](https://blog.csdn.net/weixin_45927386/article/details/142072799)]
        <br><br>
        The following is the installation process that we actually used. <br>(1080 Ti GPU w/ CUDA v12.5, worked in `torch<=2.7.0` and `causal-conv1d<=1.16.0`):
        ```bash
        git clone https://github.com/Dao-AILab/causal-conv1d.git
        cd causal-conv1d
        vim setup.py
            # press i to enter insert mode
            # add the following two lines in the proper place
            cc_flag.append("-gencode")                      # add this line
            cc_flag.append("arch=compute_60,code=sm_60")    # add this line
            cc_flag.append("-gencode")                      # original line
            cc_flag.append("arch=compute_62,code=sm_62")    # original line
            # escape and press :wq to save and exit
        CAUSAL_CONV1D_FORCE_BUILD=TRUE pip install . --no-build-isolation
        ```

In [3]:
# %pip install causal-conv1d --no-build-isolation

In [4]:
%pip show causal-conv1d

Name: causal_conv1d
Version: 1.6.0
Summary: Causal depthwise conv1d in CUDA, with a PyTorch interface
Home-page: https://github.com/Dao-AILab/causal-conv1d
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /home/yoom618/.conda/envs/py312/lib/python3.12/site-packages
Requires: ninja, packaging, torch
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
# testing causal_conv1d
from causal_conv1d import causal_conv1d_fn

causal_conv1d_fn(torch.randn(1, 16, 100).to('cuda'),
                 torch.randn(16, 3).to('cuda'),
                 bias=torch.randn(16).to('cuda'),
                 activation='silu',)

tensor([[[ 0.4250, -0.2195,  0.9923,  ...,  1.6214, -0.1500,  1.6300],
         [ 0.7266,  2.3923,  1.7577,  ...,  0.6975,  7.4288,  0.6627],
         [-0.2382, -0.2485, -0.2766,  ..., -0.1620, -0.2768, -0.2755],
         ...,
         [-0.2023,  0.9498,  0.0224,  ...,  0.1811, -0.2332,  0.1027],
         [-0.0294, -0.2579, -0.2609,  ..., -0.2241,  0.9190, -0.0574],
         [ 0.7657,  1.0435,  0.1940,  ...,  0.6137,  1.2141,  0.0450]]],
       device='cuda:0')

- As same as above, we built mamba-ssm from source code:

    ```bash
    git clone clone https://github.com/state-spaces/mamba.git
    cd mamba
    vim setup.py
        # press i to enter insert mode
        # add the following two lines in the proper place
        cc_flag.append("-gencode")                      # add this line
        cc_flag.append("arch=compute_60,code=sm_60")    # add this line
        cc_flag.append("-gencode")                      # original line
        cc_flag.append("arch=compute_62,code=sm_62")    # original line
        # escape and press :wq to save and exit
    MAMBA_FORCE_BUILD=TRUE pip install . --no-build-isolation
    ```

In [6]:
# %pip install mamba-ssm --no-build-isolation

In [7]:
%pip show mamba-ssm

Name: mamba_ssm
Version: 2.3.0
Summary: Mamba state-space model
Home-page: https://github.com/state-spaces/mamba
Author: Tri Dao, Albert Gu
Author-email: Tri Dao <tri@tridao.me>, Albert Gu <agu@cs.cmu.edu>
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      dire

In [11]:
### testing Mamba1
import torch
from mamba_ssm import Mamba

batch, length, dim = 2, 7, 16
x = torch.randn(batch, length, dim).to("cuda")
model = Mamba(
    d_model=dim, # Model dimension d_model
    d_state=16,  # SSM state expansion factor
    d_conv=4,    # Local convolution width
    expand=2,    # Block expansion factor
).to("cuda")
y = model(x)
assert y.shape == x.shape

In [12]:
# testing Mamba2
from mamba_ssm import Mamba2

batch, length, dim = 2, 7, 16
x = torch.randn(batch, length, dim).to("cuda")
model = Mamba2(
    d_model=dim, # Model dimension d_model
    d_state=64,  # SSM state expansion factor, typically 64 or 128
    d_conv=4,    # Local convolution width
    expand=2,    # Block expansion factor
    headdim=4,   # d_model * expand / headdim must be multiple of 8
).to("cuda")
y = model(x)
assert y.shape == x.shape

## Download datasets
- download datasets from [Google Drive](https://drive.google.com/drive/folders/1dJx_rpB7UnkMuxrCEoHJcXXzhaACS5Sx?usp=share_link), unzip them, and place them in the folder that you want.
- It doesn't matter where you put the datasets since you can specify the path to the datasets when you run the training or evaluation scripts.
- You can also download the checkpoints of MambaSL and other baselines from the same Google Drive link. Place the checkpoints in the folder that you want and specify the path to the checkpoints when you run the evaluation scripts.

In [ ]:
!unzip -q -o "UEA30.zip" -d "."
!mv "UEA30" "dataset"

In [ ]:
!ls dataset/ -ah

.			   FaceDetection	  PEMS-SF
..			   FingerMovements	  PenDigits
ArticularyWordRecognition  HandMovementDirection  PhonemeSpectra
AtrialFibrillation	   Handwriting		  PSM
BasicMotions		   Heartbeat		  RacketSports
CharacterTrajectories	   illness		  SelfRegulationSCP1
Cricket			   InsectWingbeat	  SelfRegulationSCP2
DuckDuckGeese		   JapaneseVowels	  SMAP
EigenWorms		   Libras		  SMD
electricity		   LSST			  SpokenArabicDigits
Epilepsy		   m4			  StandWalkJump
ERing			   __MACOSX		  SWaT
EthanolConcentration	   MotorImagery		  traffic
ETT-small		   MSL			  UWaveGestureLibrary
exchange_rate		   NATOPS		  weather
